# 🔒 Privacy Filter — Fine-tuning su Kaggle (T4)
**openai/privacy-filter** — Fine-tuning su documenti italiani + dominio legale

## ⚙️ Prerequisiti Kaggle

Prima di eseguire:
1. **Settings → Accelerator** → `GPU T4 x2` (o `GPU T4`, va bene anche una sola)
2. **Settings → Internet** → ON (serve per `pip install` e download del modello base)
3. **Session** → almeno 30 GB di RAM (il default basta — evita "NoBackend")

Se vedi `exit code -9` significa OOM RAM host: riduci `--batch-size` (già a 1 di default qui) o `--n-ctx`.

## Output
I checkpoint finali finiscono in `/kaggle/working/` — li trovi nel pannello **Output** a destra dopo il training. Da lì li scarichi sul tuo Mac.

---
## Indice
1. Setup: install + scrittura `dataset_builder.py`
2. Verifica GPU CUDA (T4)
3. Genera dataset sintetico (6 file .jsonl: train/val/test × step1/step2)
4. Training Step 1 — Documenti italiani
5. Training Step 2 — Dominio legale
6. Test qualitativo
7. Valutazione quantitativa (precision/recall/F1) su test set held-out
8. Copia checkpoint in /kaggle/working/

## 1. Setup

Installa `opf` e scrive il modulo `dataset_builder.py` in `/kaggle/working/` (inline, così il notebook è self-contained e non servono upload esterni).

In [ ]:
# Cella 1 — Installazione + scrittura dataset_builder.py
import subprocess, sys, os, base64

os.chdir('/kaggle/working')

# Install opf dal repo ufficiale (non su PyPI)
print('📦 Installazione opf...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'git+https://github.com/openai/privacy-filter.git',
                       'accelerate', 'safetensors', 'tiktoken'])
print('✅ opf installato')

# Scrivi dataset_builder.py inline (base64 per evitare collisioni con le triple quote del sorgente)
DATASET_BUILDER_B64 = (
    "..."
)
src = base64.b64decode(DATASET_BUILDER_B64).decode('utf-8')

with open('/kaggle/working/dataset_builder.py', 'w') as f:
    f.write(src)

# Aggiungi /kaggle/working al path
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

# Verifica
from dataset_builder import LABEL_SPACE, build_complete_dataset
print(f'✅ dataset_builder.py scritto ({len(LABEL_SPACE["span_class_names"])} label, {len(src):,} caratteri)')

# Verifica CLI opf
subprocess.run(['opf', '--help'], capture_output=False)


## 2. Verifica GPU CUDA

Su Kaggle con GPU T4: `DEVICE_CLI='cuda'`, la T4 ha 16 GB VRAM dedicati.

In [ ]:
# Cella 2 — Verifica device e ambiente
import torch, sys

IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in __import__('os').environ

if torch.cuda.is_available():
    DEVICE_CLI = 'cuda'
    DEVICE_HF  = 0
    print(f'✅ CUDA: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'   CUDA count: {torch.cuda.device_count()}')
else:
    DEVICE_CLI = 'cpu'
    DEVICE_HF  = -1
    print('⚠️  GPU non rilevata — verifica Settings → Accelerator → GPU T4 x2')

print(f'\nAmbiente: {"Kaggle" if IS_KAGGLE else "altro"}')
print(f'PyTorch: {torch.__version__}')


## 3. Genera dataset sintetico

Chiama `build_complete_dataset` dal modulo: produce **3400 esempi** totali (1500+250+250 step1, 1000+200+200 step2), poi li scrive in `/kaggle/working/datasets/`.

In [ ]:
# Cella 3 — Genera dataset sintetico (3400 esempi, 6 file .jsonl)
from dataset_builder import build_complete_dataset, write_splits_to_disk, validate_spans, label_distribution

bundle = build_complete_dataset(
    n_step1=(1500, 250, 250),   # train, val, test
    n_step2=(1000, 200, 200),
    seed=42,                     # riproducibilità
    negative_rate=0.15,          # 15% esempi negativi (riduce falsi positivi)
)

TRAIN_STEP1 = bundle['step1']['train']
VAL_STEP1   = bundle['step1']['val']
TEST_STEP1  = bundle['step1']['test']
TRAIN_STEP2 = bundle['step2']['train']
VAL_STEP2   = bundle['step2']['val']
TEST_STEP2  = bundle['step2']['test']

# Validazione
total_err = 0
for name, data in [('step1_train', TRAIN_STEP1), ('step1_val', VAL_STEP1), ('step1_test', TEST_STEP1),
                   ('step2_train', TRAIN_STEP2), ('step2_val', VAL_STEP2), ('step2_test', TEST_STEP2)]:
    err = validate_spans(data, name)
    total_err += err
    print(f'  {name:12s}: {len(data):>4} esempi, {err} errori')

if total_err == 0:
    print('\n✅ Tutti i dataset validi')
else:
    raise RuntimeError(f'{total_err} errori nel dataset')

# Salva file .jsonl
paths = write_splits_to_disk(bundle, base_dir='/kaggle/working/datasets')
print('\nFile salvati:')
for key, (path, n, size) in paths.items():
    print(f'  {path}  →  {n} esempi  ({size/1024:.1f} KB)')

# Distribuzione label Step 1
print('\nDistribuzione label Step 1 (train):')
for label, cnt in sorted(label_distribution(TRAIN_STEP1).items()):
    print(f'  {label:25s}: {cnt}')


## 4. Training Step 1 — Documenti italiani

Fine-tuning del modello base `openai/privacy-filter` su 1500 esempi di documenti d'identità italiani.

**Parametri Kaggle T4**: batch-size 1, epoche 15. Tempo stimato: 10-25 min.

Se vai in OOM, riduci le epoche o aggiungi `--n-ctx 256` al comando per ridurre la memoria delle attivazioni.

In [ ]:
# Cella 4 — Training Step 1
import subprocess, os, json
from dataset_builder import LABEL_SPACE

# Scrivi label space (auto-sufficiente)
LABEL_SPACE_PATH = '/kaggle/working/custom_label_space.json'
with open(LABEL_SPACE_PATH, 'w') as f:
    json.dump(LABEL_SPACE, f, indent=2)
print(f'✅ Label space scritto ({len(LABEL_SPACE["span_class_names"])} categorie)')

OUTPUT_STEP1 = '/kaggle/working/checkpoint_step1_italian_docs'
NUM_EPOCHS_STEP1 = 15
BATCH_SIZE_STEP1 = 1    # Kaggle T4 16GB VRAM — conservativo per evitare OOM

cmd = [
    'opf', 'train', '/kaggle/working/datasets/step1_train.jsonl',
    '--validation-dataset', '/kaggle/working/datasets/step1_val.jsonl',
    '--label-space-json', LABEL_SPACE_PATH,
    '--output-dir', OUTPUT_STEP1,
    '--epochs', str(NUM_EPOCHS_STEP1),
    '--batch-size', str(BATCH_SIZE_STEP1),
    '--grad-accum-steps', '4',   # effettivo batch=4 per stabilità, senza OOM
    '--device', DEVICE_CLI,
]

print('🚀 Step 1 — Training...')
print('Comando:', ' '.join(cmd))
print('=' * 60)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

if process.returncode == 0:
    print('\n✅ Step 1 completato:', OUTPUT_STEP1)
else:
    print(f'\n❌ Step 1 fallito (exit code {process.returncode})')
    if process.returncode == -9:
        print('   → Exit -9 = OOM RAM host. Prova --n-ctx 256 o meno epoche.')


## 5. Training Step 2 — Dominio legale

Parte dal checkpoint Step 1 (via `--checkpoint`) e fa fine-tuning ulteriore su 1000 esempi di atti notarili, contratti, sentenze, procure.

Tempo stimato: 8-20 min.

In [ ]:
# Cella 5 — Training Step 2 (parte da checkpoint Step 1)
import subprocess, os

if not os.path.isdir(OUTPUT_STEP1):
    raise RuntimeError(f'Step 1 non completato: {OUTPUT_STEP1}')

OUTPUT_STEP2 = '/kaggle/working/checkpoint_step2_legal'
NUM_EPOCHS_STEP2 = 12
BATCH_SIZE_STEP2 = 1

cmd = [
    'opf', 'train', '/kaggle/working/datasets/step2_train.jsonl',
    '--validation-dataset', '/kaggle/working/datasets/step2_val.jsonl',
    '--label-space-json', LABEL_SPACE_PATH,
    '--checkpoint', OUTPUT_STEP1,    # parte da Step 1
    '--output-dir', OUTPUT_STEP2,
    '--epochs', str(NUM_EPOCHS_STEP2),
    '--batch-size', str(BATCH_SIZE_STEP2),
    '--grad-accum-steps', '4',
    '--device', DEVICE_CLI,
]

print('🚀 Step 2 — Training dominio legale...')
print('Comando:', ' '.join(cmd))
print('=' * 60)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

if process.returncode == 0:
    print('\n✅ Step 2 completato:', OUTPUT_STEP2)
    FINAL_CHECKPOINT = OUTPUT_STEP2
else:
    print(f'\n❌ Step 2 fallito (exit code {process.returncode})')
    FINAL_CHECKPOINT = OUTPUT_STEP1


## 6. Test qualitativo

Esegue il modello finetuned su frasi di esempio per mostrare visivamente come riconosce le entità italiane.

In [ ]:
# Cella 6 — Test qualitativo del modello finetuned
from transformers import pipeline
import os

FINAL_CHECKPOINT = OUTPUT_STEP2 if os.path.isdir(OUTPUT_STEP2) else OUTPUT_STEP1

print(f'Caricamento modello da: {FINAL_CHECKPOINT}')
model_final = pipeline(
    task='token-classification',
    model=FINAL_CHECKPOINT,
    aggregation_strategy='simple',
    device=DEVICE_HF,
)
print('✅ Modello caricato\n')

TEST_CASES = [
    "Il sottoscritto Mario Rossi, codice fiscale RSSMRA80A01H501U, chiede il rilascio del certificato.",
    "Carta d'identità n. AB1234567, intestata a Giulia Bianchi, nata a Milano il 15/03/1990.",
    "Patente: CD456EF78901G — titolare: Luca Ferrari, residente a Roma.",
    "Bonifico su IBAN IT60X0542811101000000123456 intestato a Marco Ricci.",
    "Avanti a me, Notaio dott. Carlo Verdi, sono comparsi: il sig. Antonio Russo, C.F. RSSNTN75C15F205K.",
    "Nel procedimento n. 1234/2023 RG pendente avanti al Tribunale di Roma, tra Francesca Mancini (attore) e Roberto Costa (convenuto).",
    "L'immobile censito come foglio 45, mappale 123, sub. 7, sito nel Comune di Napoli.",
]

def redact(text, clf):
    res = sorted(clf(text), key=lambda x: x['start'], reverse=True)
    t = text
    for r in res:
        t = t[:r['start']] + f'[{r["entity_group"]}]' + t[r['end']:]
    return t

print('=' * 70)
for i, text in enumerate(TEST_CASES, 1):
    print(f'[{i:2d}] Input   : {text}')
    print(f'     Redatto : {redact(text, model_final)}')
    print()


## 7. Valutazione quantitativa sul test set held-out

Precision / Recall / F1 per ogni categoria su 450 esempi di test mai visti durante il training.

Soglia indicativa: **F1 > 0.85** per categorie principali.

In [ ]:
# Cella 7 — Metriche su test set held-out
from collections import defaultdict

def compute_span_metrics(classifier, test_data, match='exact'):
    tp = defaultdict(int); fp = defaultdict(int); fn = defaultdict(int)
    for ex in test_data:
        text = ex['text']
        gold = set()
        for label, spans in ex.get('spans', {}).items():
            for s, e in spans:
                gold.add((label, s, e))
        preds = set()
        for r in classifier(text):
            preds.add((r['entity_group'], int(r['start']), int(r['end'])))
        if match == 'exact':
            for p in preds:
                (tp if p in gold else fp)[p[0]] += 1
            for g in gold:
                if g not in preds:
                    fn[g[0]] += 1
        else:
            used_gold = set()
            for p_lab, p_s, p_e in preds:
                hit = False
                for g in gold - used_gold:
                    g_lab, g_s, g_e = g
                    if g_lab == p_lab and not (p_e <= g_s or g_e <= p_s):
                        tp[p_lab] += 1; used_gold.add(g); hit = True; break
                if not hit:
                    fp[p_lab] += 1
            for g_lab, g_s, g_e in gold - used_gold:
                fn[g_lab] += 1
    return tp, fp, fn

def print_metrics_table(tp, fp, fn, title=''):
    labels = sorted(set(tp) | set(fp) | set(fn))
    tot_tp = sum(tp.values()); tot_fp = sum(fp.values()); tot_fn = sum(fn.values())
    print(f'\n{title}')
    print(f'{"label":25s} {"TP":>5s} {"FP":>5s} {"FN":>5s} {"P":>7s} {"R":>7s} {"F1":>7s}')
    print('─' * 70)
    for l in labels:
        p = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] > 0 else 0.0
        r = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] > 0 else 0.0
        f1 = 2*p*r/(p+r) if p+r > 0 else 0.0
        marker = ' ✅' if f1 >= 0.85 else (' ⚠️ ' if f1 >= 0.7 else ' ❌')
        print(f'{l:25s} {tp[l]:>5d} {fp[l]:>5d} {fn[l]:>5d} {p:>7.3f} {r:>7.3f} {f1:>7.3f}{marker}')
    mp = tot_tp / (tot_tp + tot_fp) if tot_tp + tot_fp > 0 else 0.0
    mr = tot_tp / (tot_tp + tot_fn) if tot_tp + tot_fn > 0 else 0.0
    mf1 = 2*mp*mr/(mp+mr) if mp+mr > 0 else 0.0
    print('─' * 70)
    print(f'{"MICRO AVG":25s} {tot_tp:>5d} {tot_fp:>5d} {tot_fn:>5d} {mp:>7.3f} {mr:>7.3f} {mf1:>7.3f}')
    return mf1

print('Match EXACT (span perfettamente coincidenti):\n')
tp1, fp1, fn1 = compute_span_metrics(model_final, TEST_STEP1, match='exact')
f1_s1 = print_metrics_table(tp1, fp1, fn1, title=f'═══ TEST STEP 1 ({len(TEST_STEP1)} esempi) ═══')
tp2, fp2, fn2 = compute_span_metrics(model_final, TEST_STEP2, match='exact')
f1_s2 = print_metrics_table(tp2, fp2, fn2, title=f'\n═══ TEST STEP 2 ({len(TEST_STEP2)} esempi) ═══')

print(f'\n{"═"*70}')
print(f'F1 Step 1: {f1_s1:.3f}  |  F1 Step 2: {f1_s2:.3f}  |  F1 medio: {(f1_s1+f1_s2)/2:.3f}')
print(f'{"═"*70}')


## 8. Scarica i checkpoint

I checkpoint sono già in `/kaggle/working/` — visibili nel pannello **Output** a destra.

Per scaricarli:
1. Sulla destra, clicca sul pannello "Output"
2. Naviga in `checkpoint_step2_legal/` (il finale) o `checkpoint_step1_italian_docs/` (intermedio)
3. Download dei file `model.safetensors`, `config.json`, `finetune_summary.json`, `USAGE.txt`

Per usarli localmente sul Mac: copia l'intera cartella in `~/Documents/privacy_filter_checkpoints/step2_legal/` e caricala con `transformers.pipeline('token-classification', model='~/Documents/privacy_filter_checkpoints/step2_legal')`.

Se preferisci uno zip unico:

In [ ]:
# Cella 8 — Crea uno zip dei checkpoint per scaricarli facilmente
import shutil, os

zip_paths = []
for src, name in [(OUTPUT_STEP1, 'checkpoint_step1_italian_docs'),
                  (OUTPUT_STEP2, 'checkpoint_step2_legal')]:
    if os.path.isdir(src):
        zip_path = f'/kaggle/working/{name}.zip'
        shutil.make_archive(f'/kaggle/working/{name}', 'zip', src)
        size = os.path.getsize(zip_path) / 1e6
        zip_paths.append((zip_path, size))
        print(f'✅ {zip_path}  ({size:.1f} MB)')
    else:
        print(f'⚠️  {src} non trovato, saltato')

print('\nScarica gli zip dal pannello Output di Kaggle (pannello destro).')
